In [1]:
### Collect GDP data from the FRED API ###

# Install the required package if not already installed
%pip install fredapi
%pip install pandas pandas_datareader

"""
To use FRED API, you need to register for an API key at: https://fred.stlouisfed.org/
API documentation can be found at: https://fred.stlouisfed.org/docs/api/fred/
"""

# Import necessary libraries
import os
import pandas_datareader.data as web
import pandas as pd
from datetime import datetime
import yfinance as yf

# FRED API: Yon need to register for an API key at https://fred.stlouisfed.org/
os.environ["FRED_API_KEY"] = "7cf8b1d2d1243e21b8147cc832ff349d"
# Define the real GDP series IDs and their start dates for each country
# Note: The series IDs are based on the FRED database and may change over time.
gdp_series = {
    'United States': ('GDPC1', datetime(1960, 1, 1)),           # https://fred.stlouisfed.org/series/GDPC1 (Billions of Chained 2017 Dollars, Seasonally Adjusted Annual Rate)
}

end_date = datetime(2025, 12, 31)

# Fetch and save GDP data for each country
for country, (series_id, start_date) in gdp_series.items():
    try:
        data = web.DataReader(series_id, 'fred', start_date, end_date)
        data = data.rename(columns={series_id: 'Real GDP'})
        data.to_csv(f"quarterly_rgdp_{country.replace(' ', '_')}.csv")
        print(f"{country}: {len(data)} rows saved.")
    except Exception as e:
        print(f"Error loading {country}: {e}")
## Collect US macroeconomic data from FRED API
"""
Note that S&P 500 Index or Wilshire 5000 Total Market Index are not available in FRED now.
Instead, we collect S&P 500 index from Yahoo Finance in below cell.
"""
us_monthly_series = {
    'working_hour_manuf': 'AWHMAN',        # Avg_Weekly_Hours_Manufacturing
    'CPI': 'CPIAUCSL',                     # CPI_All_Items
    'Ids_Prd': 'INDPRO',                   # Industrial_Production
    'R_Csump': 'DPCERA3M086SBEA',          # Real_Personal_Consumption_Expenditures
    'Ef_Fed_Rate': 'FEDFUNDS',             # Effective_Fed_Funds_Rate'
    'Intr_10Y': 'GS10',                    # Interest_Rate_10Y
    'Unemp': 'UNRATE',                     # Unemployment_Rate
    'M2': 'M2SL',                          # M2_Money_Stock
    'M1': 'M1SL',                          # M1_Money_Stock
    'Labor_partic': 'CIVPART',             # Civilian_Labor_Force_Participation_Rate
    'Per_savings': 'PSAVERT',              # Personal_Savings_Rate
    'All_Emp': 'PAYEMS',                   # All_Employees_Total_Nonfarm
    'Moody_aaa': 'AAA',                    # Moody's_Aaa_Corporate_Bond_Yield_Spread
    'NetExports': 'BOPTEXP',               # Net_Exports_of_Goods_and_Services
}

start_date = datetime(1960, 1, 1)
end_date = datetime(2025, 12, 31)

monthly_data = pd.DataFrame()

for name, series_id in us_monthly_series.items():
    try:
        series = web.DataReader(series_id, 'fred', start_date, end_date)
        series = series.rename(columns={series_id: name})
        if monthly_data.empty:
            monthly_data = series
        else:
            monthly_data = monthly_data.join(series, how='outer')
        print(f"{name} loaded.")
    except Exception as e:
        print(f"Error loading {name} ({series_id}): {e}")

monthly_data = monthly_data.sort_index()
monthly_data.to_csv("us_monthly_macro_data.csv")

# Collect S&P 500 Index data from Yahoo Finance
# S&P500 100 for UK
# Ticker for S&P 500
ticker = "^GSPC"

start_date = "1960-01-01"
end_date = "2025-12-31"

# Download S&P 500 data (daily frequency)
data = yf.download(ticker, start=start_date, end=end_date, interval="1d", progress=False)

# Calculate monthly average of the 'Close' price
monthly_data = data['Close'].resample('M').mean()
monthly_data.name = "S&P500_MonthlyAvg"

monthly_data.to_csv("sp500_monthly_avg.csv")


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
United States: 264 rows saved.
working_hour_manuf loaded.
CPI loaded.
Ids_Prd loaded.
R_Csump loaded.
Ef_Fed_Rate loaded.
Intr_10Y loaded.
Unemp loaded.
M2 loaded.
M1 loaded.
Labor_partic loaded.
Per_savings loaded.
All_Emp loaded.
Moody_aaa loaded.
NetExports loaded.


/var/folders/tl/jsjpn16s0q174rrp839gnfch0000gn/T/ipykernel_73317/399627072.py:92: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly_data = data['Close'].resample('M').mean()


In [2]:
## Collect US trade/tariff data from FRED API (Quarterly)
us_quarterly_series = {
    'Customs_Duties': 'B235RC1Q027SBEA',   # Federal_govt_tax_receipts_Customs_duties
    'Imports_Goods': 'A255RC1Q027SBEA',    # Imports_of_goods
    'GDP_nominal': 'GDP',                   # Gross_Domestic_Product
}

start_date = datetime(1960, 1, 1)
end_date = datetime(2025, 12, 31)

quarterly_data = pd.DataFrame()

for name, series_id in us_quarterly_series.items():
    try:
        series = web.DataReader(series_id, 'fred', start_date, end_date)
        series = series.rename(columns={series_id: name})
        if quarterly_data.empty:
            quarterly_data = series
        else:
            quarterly_data = quarterly_data.join(series, how='outer')
        print(f"{name} loaded.")
    except Exception as e:
        print(f"Error loading {name} ({series_id}): {e}")

quarterly_data = quarterly_data.sort_index()
quarterly_data.to_csv("us_quarterly_trade_data.csv")

Customs_Duties loaded.
Imports_Goods loaded.
GDP_nominal loaded.


In [3]:
### Merge US Data
import pandas as pd

# Load datasets
us_gdp_df = pd.read_csv("quarterly_rgdp_United_States.csv", parse_dates=["DATE"]) # From FRED API
us_macro_df = pd.read_csv("us_monthly_macro_data.csv", parse_dates=["DATE"])      # From FRED API
sp500_df = pd.read_csv("sp500_monthly_avg.csv", parse_dates=["Date"])             # From Yahoo Finance API
us_quart_trade = pd.read_csv("us_quarterly_trade_data.csv", parse_dates=["DATE"])

# Rename columns for consistency
sp500_df = sp500_df.rename(columns={"Date": "DATE", "S&P500_MonthlyAvg": "S&P500"})
sp500_df["DATE"] = sp500_df["DATE"].dt.to_period("M").dt.to_timestamp()

# Convert GDP DATE to month start (quarter start)
us_gdp_df["DATE"] = us_gdp_df["DATE"].dt.to_period("M").dt.to_timestamp()

# Convert quarterly trade DATE to month start (quarter start)
us_quart_trade["DATE"] = us_quart_trade["DATE"].dt.to_period("M").dt.to_timestamp()

# Merge monthly data
us_monthly_merged = pd.merge(us_macro_df, sp500_df, on="DATE", how="outer")

# Merge with quarterly GDP data (Real GDP will appear only at quarter start months)
us_full_merged = pd.merge(us_monthly_merged, us_gdp_df, on="DATE", how="left")

# Merge with quarterly trade data (Trade data will appear only at quarter start months)
us_full_merged = pd.merge(us_full_merged, us_quart_trade, on="DATE", how="left")

us_full_merged = us_full_merged.sort_values("DATE").reset_index(drop=True)

# Save the merged DataFrame to CSV
us_full_merged.to_csv("master_US.csv", index=False)

In [6]:
%pip install scikit-learn
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error


dataset = pd.read_csv("master_US.csv")
dataset = dataset.sort_values("DATE").reset_index(drop=True)

# Fix: drop Real GDP with correct name
dataset = dataset.drop(columns=["DATE", "Real GDP", "NetExports"])

# Fix: drop rows where key columns are missing FIRST
dataset = dataset.dropna(subset=["GDP_nominal", "Customs_Duties", "Imports_Goods"])

print("After cleaning:", dataset.shape)
print("\nNulls per column:")
print(dataset.isna().sum().sort_values(ascending=False))

# Now shift and lag
dataset["GDP_nominal"] = dataset["GDP_nominal"].shift(-1)

lag_cols = ["CPI", "M2", "R_Csump", "Unemp"]
for col in lag_cols:
    dataset[f"{col}_lag1"] = dataset[col].shift(1)

dataset = dataset.dropna()
print("Final shape:", dataset.shape)

# Split features and label
X = dataset.drop(columns=["GDP_nominal"])
y = dataset["GDP_nominal"]

# Train model
model = GradientBoostingRegressor(
    n_estimators=600,
    max_depth=3,
    learning_rate=0.02,
    min_samples_leaf=10,
    random_state=42
)

model.fit(X, y)

# Print summary stats
preds = model.predict(X)
print(f"R² Score: {r2_score(y, preds):.4f}")
print(f"RMSE: {root_mean_squared_error(y, preds):.4f}")
print(f"\nFeature Importances:")
for feat, imp in sorted(zip(X.columns, model.feature_importances_), key=lambda x: -x[1]):
    print(f"  {feat}: {imp:.4f}")


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
After cleaning: (264, 17)

Nulls per column:
Unemp                 1
CPI                   1
Labor_partic          1
working_hour_manuf    0
Per_savings           0
Imports_Goods         0
Customs_Duties        0
^GSPC                 0
Moody_aaa             0
All_Emp               0
M1                    0
M2                    0
Intr_10Y              0
Ef_Fed_Rate           0
R_Csump               0
Ids_Prd               0
GDP_nominal           0
dtype: int64
Final shape: (262, 21)
R² Score: 0.9999
RMSE: 67.9687

Feature Importances:
  M1: 0.2025
  M2_lag1: 0.1499
  R_Csump: 0.1409
  CPI_lag1: 0.1259
  CPI: 0.1049
  R_Csump_lag1: 0.0897
  M2: 0.0753
  Imports_Goods: 0.0447
  ^GSPC: 0.0306
  All_Emp: 0.0133
  Ids_Prd: 0.0086
  Labor_partic: 0.0086
  Customs_Duties: 0.0036
  Intr_10Y: 0.0006
  Ef_Fed_Rate: 0.

In [8]:
import numpy as np
predictions = model.predict(X)
actual = dataset["GDP_nominal"].values
mape = np.mean(np.abs((actual - predictions) / actual)) * 100
print(f"MAPE: {mape:.2f}%")
print(f"RMSE: {np.sqrt(np.mean((actual - predictions)**2)):.2f}")

MAPE: 0.52%
RMSE: 67.97


In [13]:
# Load necessary libraries
%pip install statsmodels
import os, random
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller    # For ADF test
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras                      # TensorFlow does not support Python 3.12 yet (as of June 2025)
%pip install keras-tuner
%pip install seaborn
import keras_tuner as kt
import matplotlib.pyplot as plt
import seaborn as sns


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Obtaining dependency information for seaborn from https://files.pythonhosted.org/packages/83/11/00d3c3dfc25ad54e731d91449895a79e4bf2384dc3ac01809010ba88f6d5/seaborn-0.13.2-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 5.8 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
## Algorithm 1 & 2: Data Preparation and Quarterly Aggregation of Variables

# 1. Load the master dataset
df = pd.read_csv('master_US.csv')

# 2. Parse the DATE column and create a quarter identifier
df['DATE'] = pd.to_datetime(df['DATE'])
df['quarter'] = df['DATE'].dt.to_period('Q')

# 3. Specify the target variable name
Y_col = 'Real GDP'

# 5. Identify explanatory variables (exclude DATE, quarter, and Y column)
exclude_cols = {'DATE', 'quarter', Y_col}
x_cols = [col for col in df.columns if col not in exclude_cols]

# 6. Run ADF test for each explanatory variable
adf_results = pd.DataFrame(columns=["Variable", "ADF Statistic", "p-value", "Stationary at 5%"])
X_prime_m_df = df[['DATE', 'quarter']].copy()  # To store transformed monthly variables

for col in x_cols:
    series_original = df[col].copy()
    series_numeric = pd.to_numeric(series_original, errors='coerce').dropna()

    if not series_numeric.empty:
        result = adfuller(series_numeric)
        adf_stat = result[0]
        p_value = result[1]
        is_stationary = p_value < 0.05
        adf_results.loc[len(adf_results)] = [col, adf_stat, p_value, is_stationary]
    else:
        adf_results.loc[len(adf_results)] = [col, np.nan, np.nan, False]

# Display ADF test results
adf_results_display = adf_results.set_index("Variable").sort_values("p-value")
print("\n=== ADF Test Results ===")
print(adf_results_display)


=== ADF Test Results ===
                    ADF Statistic   p-value  Stationary at 5%
Variable                                                     
working_hour_manuf      -3.527745  0.007302              True
Unemp                   -3.397147  0.011057              True
Ef_Fed_Rate             -2.978202  0.036981              True
Per_savings             -2.551066  0.103538             False
Moody_aaa               -1.605166  0.480998             False
Labor_partic            -1.455126  0.555582             False
Ids_Prd                 -1.391327  0.586427             False
Intr_10Y                -1.241111  0.655666             False
All_Emp                 -0.743837  0.835066             False
NetExports               0.081512  0.964776             False
M1                       0.879478  0.992817             False
Imports_Goods            1.998030  0.998670             False
CPI                      2.068364  0.998755             False
M2                       3.042097  1.000000 

In [15]:
# 7. Manually specify which X columns should be log-differenced
log_diff_cols = [
    'Ids_Prd',
    'R_Csump',
    'All_Emp',
    'NetExports',
    'M1',
    'M2',
    'CPI',
    '^GSPC',
    'Intr_10Y',
    'Moody_aaa',
    'Labor_partic',
    'Imports_Goods',
    'Customs_Duties',
    'GDP_nominal'
]

# 8. Identify explanatory variables that are not log-differenced
non_log_cols = [col for col in x_cols if col not in log_diff_cols]

# 9. Apply transformation to create X_prime_m_df (transformed monthly X)
for col in x_cols:
    if col in log_diff_cols:
        if (df[col] <= 0).any():
            print(f"Warning: Column {col} contains non-positive values. Applying log.diff() may result in NaNs or errors.")
            try:
                X_prime_m_df[col] = np.log(df[col].replace(0, np.nan)).diff()
            except Exception as e:
                print(f"Error log-differencing {col}: {e}. Column will be NaN.")
                X_prime_m_df[col] = np.nan
        else:
            X_prime_m_df[col] = np.log(df[col]).diff()
    else:
        X_prime_m_df[col] = df[col].copy()


# 10. Group the transformed monthly X by quarter
quarter_groups_X_prime = X_prime_m_df.groupby('quarter')

# 11. For log-differenced variables, take the quarterly sum
x_q_aggregated_log = quarter_groups_X_prime[log_diff_cols].sum() if log_diff_cols else pd.DataFrame(index=quarter_groups_X_prime.size().index)

# 12. For non-log-differenced variables, take the quarterly mean
x_q_aggregated_nonlog = quarter_groups_X_prime[non_log_cols].mean() if non_log_cols else pd.DataFrame(index=quarter_groups_X_prime.size().index)

# 13. Combine aggregated quarterly explanatory variables
X_q_aggregated = pd.concat([x_q_aggregated_log, x_q_aggregated_nonlog], axis=1).sort_index()

# 14. Extract quarterly target observations (use the first month of each quarter)
y_q_levels_df = (
    df[df[Y_col].notna()][['quarter', Y_col]]
    .drop_duplicates(subset='quarter', keep='first')    # Ensure we take the first month of each quarter
    .set_index('quarter')
    .sort_index()
)

# 15. Align aggregated X with quarterly Y
common_index = X_q_aggregated.index.intersection(y_q_levels_df.index)
X_q_aligned = X_q_aggregated.loc[common_index]
Y_q_levels = y_q_levels_df.loc[common_index, Y_col]

# 16. Convert Y to quarterly log difference (growth rate)
Y_q_processed = np.log(Y_q_levels).diff().dropna()

# 17. Align X again with the log-differenced Y (drops one more quarter)
X_q_processed = X_q_aligned.loc[Y_q_processed.index]

# Display
print("\n--- Final Processed Data (Before Train/Test Split, No scaling for non-log) ---")
print("Processed Quarterly X shape:", X_q_processed.shape)
print("Processed Quarterly Y shape:", Y_q_processed.shape)
print("\nX_q_processed (first few rows):")
print(X_q_processed.head())
print("\nY_q_processed (first few rows):")
print(Y_q_processed.head())


--- Final Processed Data (Before Train/Test Split, No scaling for non-log) ---
Processed Quarterly X shape: (263, 18)
Processed Quarterly Y shape: (263,)

X_q_processed (first few rows):
          Ids_Prd   R_Csump   All_Emp  NetExports        M1        M2  \
quarter                                                                 
1960Q2  -0.021751 -0.005299 -0.001948         0.0 -0.001432  0.009973   
1960Q3  -0.015162  0.005859 -0.002229         0.0  0.011396  0.019978   
1960Q4  -0.034672 -0.010997 -0.008984         0.0 -0.003547  0.012887   
1961Q1   0.006067  0.018488 -0.001546         0.0  0.008493  0.018710   
1961Q2   0.049536  0.008965  0.005909         0.0  0.007023  0.018675   

              CPI     ^GSPC  Intr_10Y  Moody_aaa  Labor_partic  Imports_Goods  \
quarter                                                                         
1960Q2   0.006777  0.040048 -0.023811  -0.008949      0.020305            0.0   
1960Q3   0.000000 -0.043916 -0.088107  -0.045985      0.0

In [16]:
## Algorithm 3: Neural Network Hyperparameter Optimization

# Set Seed for Reproducibility
os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

# 1. Split the data into training, test, validation (unseen) sets (70/20/10 split)

# Sort by date
X_q_processed = X_q_processed.sort_index()
Y_q_processed = Y_q_processed.sort_index()

n = len(X_q_processed)
train_end = int(n*0.7)
test_end = int(n*0.9)

X_q_train = X_q_processed.iloc[:train_end]
Y_q_train = Y_q_processed.iloc[:train_end]
X_q_test = X_q_processed.iloc[train_end:test_end]
Y_q_test = Y_q_processed.iloc[train_end:test_end]
X_q_val = X_q_processed.iloc[test_end:]
Y_q_val = Y_q_processed.iloc[test_end:]

# 2. Standardize non-log-differenced columns
scaler = StandardScaler()    # Z-score
X_q_train_scaled = X_q_train.copy()
X_q_test_scaled = X_q_test.copy()
if non_log_cols:
    X_q_train_scaled[non_log_cols] = scaler.fit_transform(X_q_train[non_log_cols])  # For training data, fit the scaler
    X_q_test_scaled[non_log_cols] = scaler.transform(X_q_test[non_log_cols])        # For test data, we do not fit again, just transform to avid data leakage

# Sign accuracy funtion
def sign_acc_metric(y_true, y_pred):
    return tf.reduce_mean(tf.cast(tf.equal(tf.sign(y_true), tf.sign(y_pred)), tf.float32))

In [17]:
train_df = pd.concat([X_q_train, Y_q_train], axis=1)
test_df  = pd.concat([X_q_test, Y_q_test], axis=1)
val_df   = pd.concat([X_q_val, Y_q_val], axis=1)

# Add a column to identify the split
train_df["dataset"] = "train"
test_df["dataset"]  = "test"
val_df["dataset"]   = "validation"

# Combine all into one dataframe
final_df = pd.concat([train_df, test_df, val_df])

# Save to CSV
final_df.to_csv("all_splits.csv", index=False)

In [18]:
# 2. Standardize non-log-differenced columns
scaler = StandardScaler()    # Z-score
X_q_train_scaled = X_q_train.copy()
X_q_test_scaled = X_q_test.copy()
if non_log_cols:
    X_q_train_scaled[non_log_cols] = scaler.fit_transform(X_q_train[non_log_cols])  # For training data, fit the scaler
    X_q_test_scaled[non_log_cols] = scaler.transform(X_q_test[non_log_cols])        # For test data, we do not fit again, just transform to avoid data leakage

# Sign accuracy funtion
def sign_acc_metric(y_true, y_pred):
    return tf.reduce_mean(tf.cast(tf.equal(tf.sign(y_true), tf.sign(y_pred)), tf.float32))

# 3. KerasTuner funtion
def build_model(hp):
    model = keras.Sequential()
    model.add(keras.layers.Input(shape=(X_q_train_scaled.shape[1],)))
    for i in range(hp.Int('num_layers', 1, 5)):  # up to 5 layers (may increase for more complexity)
        model.add(
            keras.layers.Dense(
                units=hp.Int(f'units_{i}', min_value=8, max_value=516, step=8),
                activation=hp.Choice('activation', ['relu', 'tanh', 'elu', 'selu', 'swish'])
            )
        )
        model.add(
            keras.layers.Dropout(
                rate=hp.Float(f'dropout_{i}', min_value=0.0, max_value=0.5, step=0.05)
            )
        )
    model.add(keras.layers.Dense(1, activation='linear'))
    model.compile(
        optimizer=keras.optimizers.Adam(         # Adam optimizer
            learning_rate=hp.Choice('lr', [1e-2, 5e-3, 1e-3, 5e-4, 1e-4])
        ),
        loss='mse',
        metrics=['mse', sign_acc_metric]
    )
    return model

tuner = kt.BayesianOptimization(
    build_model,
    objective='val_mse',
    max_trials=100,                              # Increase trials for better search
    executions_per_trial=1,
    overwrite=True,
    directory='keras_tuner_dir',
    project_name='gdp_q_nn_wide'
)

tuner.search(
    X_q_train_scaled.values, Y_q_train.values,
    epochs=2000,
    batch_size=4,
    validation_data=(X_q_test_scaled.values, Y_q_test.values),
    callbacks=[keras.callbacks.EarlyStopping(patience=200, restore_best_weights=True)],
    verbose=2
)

# Get the best hyperparameters and model
best_hp = tuner.get_best_hyperparameters(1)[0]
best_model = tuner.get_best_models(1)[0]

print("\n[Best Hyperparameters]")
for k,v in best_hp.values.items():
    print(f"{k}: {v}")

Trial 73 Complete [00h 14m 00s]
val_mse: 1.747128408169374e-05

Best val_mse So Far: 1.4834585272183176e-05
Total elapsed time: 01h 48m 27s

Search: Running Trial #74

Value             |Best Value So Far |Hyperparameter
5                 |5                 |num_layers
168               |104               |units_0
swish             |selu              |activation
0.15              |0.2               |dropout_0
0.0001            |0.001             |lr
504               |88                |units_1
0.2               |0.1               |dropout_1
184               |336               |units_2
0                 |0.05              |dropout_2
352               |488               |units_3
0.3               |0.45              |dropout_3
216               |128               |units_4
0.3               |0.05              |dropout_4

Epoch 1/2000
46/46 - 1s - 26ms/step - loss: 1.8851e-04 - mse: 1.8851e-04 - sign_acc_metric: 0.7446 - val_loss: 4.6922e-04 - val_mse: 4.6922e-04 - val_sign_acc_metric: 0.

KeyboardInterrupt: 

In [ ]:
# 5. Quarterly Prediction and Evaluation
y_test_pred = best_model.predict(X_q_test_scaled.values).flatten()

# Define functions for sMAPE and Theil's U1
def smape(y_true, y_pred):
    """Symmetric Mean Absolute Percentage Error"""
    numerator = 2 * np.abs(y_pred - y_true)
    denominator = np.abs(y_true) + np.abs(y_pred)
    ratio = np.divide(numerator, denominator, out=np.zeros_like(numerator, dtype=float), where=denominator!=0)
    return np.mean(ratio) * 100

def theil_u1(y_true, y_pred):
    """Theil's U1 statistic for forecast accuracy"""
    rmse = np.sqrt(np.mean((y_true - y_pred)**2))
    rms_actual = np.sqrt(np.mean(y_true**2))
    rms_pred = np.sqrt(np.mean(y_pred**2))
    if (rms_actual + rms_pred) == 0:
        return np.nan
    return rmse / (rms_actual + rms_pred)

# Calculate all metrics
mse = mean_squared_error(Y_q_test, y_test_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(Y_q_test, y_test_pred)
r2 = r2_score(Y_q_test, y_test_pred)
sign_acc = (np.sign(y_test_pred) == np.sign(Y_q_test)).mean()
smape_val = smape(Y_q_test, y_test_pred)
theil_u1_val = theil_u1(Y_q_test, y_test_pred)

# Print all metrics
print("\n[Model Evaluation on Test Set]")
print(f"Test MSE: {mse:.6f}")
print(f"Test RMSE: {rmse:.6f}")
print(f"Test MAE: {mae:.6f}")
print(f"Test R²: {r2:.3f}")
print(f"Sign Accuracy: {sign_acc:.2%}")
print(f"sMAPE: {smape_val:.2f}%")
print(f"Theil's U1: {theil_u1_val:.3f}")


# Create a DataFrame for per-prediction results
results_index = Y_q_test.index
results_df = pd.DataFrame({
    "Quarter": results_index.to_timestamp(),
    "Actual_Y_q": Y_q_test,
    "Predicted_Y_q": y_test_pred,
    "Sign_Match": (np.sign(y_test_pred) == np.sign(Y_q_test)).astype(int)
})
results_df.to_csv("test_MLP_US.csv", index=False)
print("\n'test_MLP.csv' saved.")

# Create and save a DataFrame for the summary metrics
metrics_summary_df = pd.DataFrame({
    'Metric': ['MSE', 'RMSE', 'MAE', 'R2', 'Sign Accuracy', 'sMAPE', 'Theil U1'],
    'Value': [mse, rmse, mae, r2, sign_acc, smape_val, theil_u1_val]
})
metrics_summary_df.to_csv("test_MLP_US_summary_metrics.csv", index=False)
print("'test_MLP_US_summary_metrics.csv' saved.")

# Plotting
sns.set_theme(style="whitegrid", font_scale=1.2, rc={"lines.linewidth": 2.5})

plt.figure(figsize=(12, 5))
sns.lineplot(x=Y_q_test.index.to_timestamp(), y=Y_q_test.values, label="Actual Y_q", marker="o")
sns.lineplot(x=Y_q_test.index.to_timestamp(), y=y_test_pred, label="Predicted Y_q", marker="X")

plt.title("Test Set Prediction vs Actual (Quarterly)", fontsize=16, weight='bold')
plt.xlabel("Quarter")
plt.ylabel("Log-diff Growth")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 6. Monthly GDP prediction with reconciliation (Proportional Denton)

# Monthly X scaling (matching columns and order used in model training)
X_m_scaled = X_prime_m_df[X_q_train.columns].copy()
if non_log_cols:
    X_m_scaled[non_log_cols] = scaler.transform(X_m_scaled[non_log_cols])

# Monthly "quarterly-perspective" growth prediction
monthly_y_pred_nn = best_model.predict(X_m_scaled.values).flatten()
monthly_predictions_df = pd.DataFrame({
    'raw_quarterly_log_diff_pred': monthly_y_pred_nn
}, index=X_prime_m_df['DATE'])
monthly_predictions_df.index = pd.to_datetime(monthly_predictions_df.index)
monthly_predictions_df['quarter'] = monthly_predictions_df.index.to_period('Q')

# Naive monthly growth: simply distribute predicted quarterly growth equally to each month
monthly_predictions_df['naive_monthly_log_diff'] = monthly_predictions_df['raw_quarterly_log_diff_pred'] / 3.0

# Actual observed quarterly log growth (Y_q_processed)
actual_quarterly_log_diffs = Y_q_processed.copy()
monthly_predictions_df['quarter'] = monthly_predictions_df.index.to_period('Q')

# For each quarter, calculate scaling factor so the sum of naive monthly growths matches actual quarterly growth
quarterly_sum_of_naive = monthly_predictions_df.groupby('quarter')['naive_monthly_log_diff'].sum()
adjusted_monthly_log_diffs = pd.Series(index=monthly_predictions_df.index, dtype=float)

for quarter, group in monthly_predictions_df.groupby('quarter'):
    if quarter in actual_quarterly_log_diffs.index and quarter in quarterly_sum_of_naive.index:
        actual_q_log_diff = actual_quarterly_log_diffs.loc[quarter]
        sum_naive = quarterly_sum_of_naive.loc[quarter]
        if sum_naive != 0:
            adjustment_factor = actual_q_log_diff / sum_naive
            adjusted_monthly_log_diffs.loc[group.index] = group['naive_monthly_log_diff'] * adjustment_factor
        else:
            # If sum_naive is zero, evenly distribute actual growth across the months in the quarter
            adjusted_monthly_log_diffs.loc[group.index] = actual_q_log_diff / len(group)
    else:
        # If we have no actual value, just use the naive value
        adjusted_monthly_log_diffs.loc[group.index] = group['naive_monthly_log_diff']

monthly_predictions_df['adjusted_monthly_log_diff'] = adjusted_monthly_log_diffs

# Export predicted monthly GDP growth as CSV
mgdp_pred_df = pd.DataFrame({
    "MGDP_logdiff_pred": monthly_predictions_df['adjusted_monthly_log_diff']
}, index=monthly_predictions_df.index)
mgdp_pred_df.index.name = "DATE"
mgdp_pred_df.to_csv("US_monthly_GDP_MLP_growth.csv")
print("US_monthly_GDP_MLP_growth.csv saved")


In [ ]:
# Plotting Predicted Monthly GDP Growth vs Actual Quarterly GDP Growth
monthly_pred = monthly_predictions_df['adjusted_monthly_log_diff']
quarter_to_growth = actual_quarterly_log_diffs
monthly_actual = monthly_predictions_df['quarter'].map(quarter_to_growth)

sns.set_theme(style="whitegrid", font_scale=1.2)

plt.figure(figsize=(15, 6))
sns.lineplot(x=monthly_pred.index, y=monthly_pred.values, label="Predicted Monthly GDP Growth", marker='o', linewidth=2.2)
plt.step(monthly_actual.index, monthly_actual.values, label="Actual Quarterly GDP Growth",
         where='mid', linestyle='--', color='orange', linewidth=1.8, alpha=0.7)

plt.title("Predicted Monthly GDP Growth and Actual Quarterly GDP Growth", fontsize=16, weight='bold')
plt.xlabel("Date")
plt.ylabel("Log Growth")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 7. Recover Monthly Level

# Find the first valid index (start month) for growth reconstruction
first_valid_month_idx = monthly_predictions_df['adjusted_monthly_log_diff'].first_valid_index()
first_quarter = monthly_predictions_df.loc[first_valid_month_idx, 'quarter']
base_level_quarter = first_quarter - 1

# Set base_level as the previous quarter's GDP level
if base_level_quarter in Y_q_levels.index:
    base_level = Y_q_levels.loc[base_level_quarter]
    print(f"Using GDP level of base quarter {base_level_quarter} as base_level: {base_level}")
else:
    base_level = Y_q_levels.iloc[0]
    print(f"Base quarter not found. Using the first available GDP level {Y_q_levels.index[0]}: {base_level}")

# Recover monthly GDP levels from growth (starting from the first valid growth)
reconstructed_levels = {}
current_level = base_level
for i, month in enumerate(monthly_predictions_df.index):
    if month < first_valid_month_idx:
        reconstructed_levels[month] = np.nan
    elif month == first_valid_month_idx:
        reconstructed_levels[month] = current_level
    else:
        log_diff = monthly_predictions_df.loc[month, 'adjusted_monthly_log_diff']
        if pd.isna(log_diff):
            reconstructed_levels[month] = np.nan
        else:
            prev_month = monthly_predictions_df.index[monthly_predictions_df.index.get_loc(month) - 1]
            prev_level = reconstructed_levels[prev_month]
            if pd.isna(prev_level):
                reconstructed_levels[month] = np.nan
            else:
                reconstructed_levels[month] = prev_level * np.exp(log_diff)
monthly_predictions_df['reconstructed_monthly_level'] = pd.Series(reconstructed_levels)


# Aggregate to Quarterly Level & Compare with Actual (using first month of each quarter)
monthly_predictions_df['quarter'] = monthly_predictions_df.index.to_period('Q')
re_agg_quarterly = (
    monthly_predictions_df
    .groupby('quarter')['reconstructed_monthly_level']
    .first()
)

# Convert index to Timestamp for plotting
re_agg_quarterly_ts = re_agg_quarterly.copy()
re_agg_quarterly_ts.index = re_agg_quarterly_ts.index.to_timestamp(how='start')
Y_q_levels_ts = Y_q_levels.copy()
Y_q_levels_ts.index = Y_q_levels_ts.index.to_timestamp(how='start')

comparison_df = pd.DataFrame({
    'Actual_Quarterly_Level': Y_q_levels_ts,
    'ReAgg_Monthly_Level': re_agg_quarterly_ts
}).dropna()

print("Final quarterly comparison results:")
print(comparison_df.head())
print("\nR²: %.4f, MSE: %.2f" % (
    r2_score(comparison_df['Actual_Quarterly_Level'], comparison_df['ReAgg_Monthly_Level']),
    mean_squared_error(comparison_df['Actual_Quarterly_Level'], comparison_df['ReAgg_Monthly_Level'])
))


# Plot
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10,6))
sns.lineplot(x=comparison_df.index, y=comparison_df['Actual_Quarterly_Level'], marker='o', label='Actual Quarterly Level')
sns.lineplot(x=comparison_df.index, y=comparison_df['ReAgg_Monthly_Level'], marker='x', label='ReAgg. Monthly Level')
plt.title("Actual vs. Re-Aggregated (from Monthly) Quarterly GDP Level")
plt.xlabel("Quarter")
plt.ylabel("GDP Level")
plt.legend()
plt.tight_layout()
plt.show()


# Save monthly GDP growth and level to CSV
out_df = pd.DataFrame({
    "MGDP_logdiff_pred": monthly_predictions_df['adjusted_monthly_log_diff'],
    "MGDP_level_pred": monthly_predictions_df['reconstructed_monthly_level']
}, index=monthly_predictions_df.index)
out_df.index.name = "DATE"
out_df.to_csv("US_monthly_GDP_MLP_growth_level.csv")
print("US_monthly_GDP_MLP_growth_level.csv saved!")
